In [1]:
# imports
# If these fail, please check you're running from an 'activated' environment with (llms) in the command prompt

import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_links, fetch_website_contents
from openai import OpenAI

In [3]:
# Initialize and constants

load_dotenv(override=True)
api_key = os.getenv('GOOGLE_API_KEY')

if api_key and api_key.startswith('AQ.') and len(api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")
    
MODEL = 'gemini-3.5-flash-lite'
gemini=OpenAI(base_url="https://generativelanguage.googleapis.com/v1beta/openai/",api_key=api_key)

API key looks good so far


In [4]:
links = fetch_website_links("https://www.raghuenggcollege.com/")
links

['#content',
 'https://alumni.raghuenggcollege.com/',
 '/aicte/',
 '/naac/',
 '/iqac/',
 '/iic/',
 '/nba/',
 '/nptel/',
 '/documents/Report-NIRF2026.pdf',
 '/iso-certificates/',
 '/iirs-isro-home/',
 'https://raghuenggcollege.com/ict-tools/',
 'https://raghuenggcollege.com/mat-lab/',
 'https://raghuenggcollege.com/abc/',
 'https://www.raghuenggcollege.com/raghu-bcde/',
 'https://www.raghuenggcollege.com/fee-e-payment/',
 '/dtbu/',
 'tel:+918712497101',
 'https://raghuenggcollege.com/',
 'https://www.raghuenggcollege.com/naac-3/',
 'https://www.raghuenggcollege.com/ugc/',
 'https://www.raghuenggcollege.com/nba-2/',
 'https://www.raghuenggcollege.com/documents/permanent-affiliation2024-25-to-2026-27.pdf',
 'https://raghuenggcollege.com/aicte-approvals/',
 'https://www.raghuenggcollege.com/iso-certificates/',
 '#',
 '#',
 'https://www.raghuenggcollege.com/',
 'https://www.raghuenggcollege.com/who-we-are/',
 'https://www.raghuenggcollege.com/acc-rec/',
 'https://www.raghuenggcollege.com/ou

In [5]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

In [6]:

def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [7]:
print(get_links_user_prompt("https://www.raghuenggcollege.com/"))


Here is the list of links on the website https://www.raghuenggcollege.com/ -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

#content
https://alumni.raghuenggcollege.com/
/aicte/
/naac/
/iqac/
/iic/
/nba/
/nptel/
/documents/Report-NIRF2026.pdf
/iso-certificates/
/iirs-isro-home/
https://raghuenggcollege.com/ict-tools/
https://raghuenggcollege.com/mat-lab/
https://raghuenggcollege.com/abc/
https://www.raghuenggcollege.com/raghu-bcde/
https://www.raghuenggcollege.com/fee-e-payment/
/dtbu/
tel:+918712497101
https://raghuenggcollege.com/
https://www.raghuenggcollege.com/naac-3/
https://www.raghuenggcollege.com/ugc/
https://www.raghuenggcollege.com/nba-2/
https://www.raghuenggcollege.com/documents/permanent-affiliation2024-25-to-2026-27.pdf
https://raghuenggcollege.com/aicte-approvals/
https://www.raghuenggcoll

In [18]:
def select_relevent_links(url):
    response=gemini.chat.completions.create(model="gemini-3.5-flash-lite",messages=[
        {"role":"system","content":link_system_prompt},
        {"role":"user","content":get_links_user_prompt(url)}
    ],
    response_format={"type":"json_object"}
    )
    result=response.choices[0].message.content
    links=json.loads(result)
    return links


In [19]:
select_relevent_links("https://www.raghuenggcollege.com/")

{'links': [{'type': 'about page',
   'url': 'https://www.raghuenggcollege.com/who-we-are/'},
  {'type': 'leadership page',
   'url': 'https://www.raghuenggcollege.com/our-leaders/'},
  {'type': 'leadership page',
   'url': 'https://www.raghuenggcollege.com/chairman-desk/'},
  {'type': 'leadership page',
   'url': 'https://www.raghuenggcollege.com/principal-message/'},
  {'type': 'courses page', 'url': 'https://www.raghuenggcollege.com/courses/'},
  {'type': 'departments page',
   'url': 'https://www.raghuenggcollege.com/cse-dept/'},
  {'type': 'departments page',
   'url': 'https://www.raghuenggcollege.com/ece-dept/'},
  {'type': 'departments page',
   'url': 'https://www.raghuenggcollege.com/mech-dept/'},
  {'type': 'departments page',
   'url': 'https://www.raghuenggcollege.com/civil-dept/'},
  {'type': 'departments page',
   'url': 'https://www.raghuenggcollege.com/eee-dept/'},
  {'type': 'research page',
   'url': 'https://www.raghuenggcollege.com/research/'},
  {'type': 'placement

In [23]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevent_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents(link["url"])
    return result

In [24]:
print(fetch_page_and_all_relevant_links("https://www.raghuenggcollege.com/"))

## Landing Page:

Home  - Raghu Engineering College - Autonomous | Vishakhapatnam

Skip to content
ALUMNI
AICTE
NAAC
IQAC
IIC
NBA
NPTEL
NIRF
ISO
IIRS_ISRO
ICT_TOOLS
MATLAB
ABC
BCDE
E_FEE_PAYMENTS
DTBU
Help_Desk_91_8712497101
RAGHU
ENGINEERING COLLEGE
(AUTONOMOUS | VISAKHAPATNAM)
Accredited by NAAC With ‘A+’ Grade
AUTONOMOUS UGC 12(B) and 2(F)
Reaccredited by NBA to CIVIL, MECH, ECE & CSE
Permanently Affiliated by JNTU-GV Vizianagaram
Approved by All India Council for Technical Education (AICTE)
Certified by ISO 14001:2015 & ISO 9001:2015
College Code:
RAGU
Admissions
+91 9515127092
+91 9885766942
+91 08922 248001/2
Home
Who We Are
Accreditations & Recognitions
Vision & Mission
Leadership
Governing Body
Academic Council
Organization Chart
Academics
Principal
Programs
Committees
Statutory Committees
Non-Statutory Committees
Academic Year Calendars
Departments
Computer Science & Engineering
CSE (AIML)
CSE (Data Science)
CSE (Cyber Security)
CSE (IoT)
Electronics & Communications Engineeri

In [25]:
brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""

# Or uncomment the lines below for a more humorous brochure - this demonstrates how easy it is to incorporate 'tone':

# brochure_system_prompt = """
# You are an assistant that analyzes the contents of several relevant pages from a company website
# and creates a short, humorous, entertaining, witty brochure about the company for prospective customers, investors and recruits.
# Respond in markdown without code blocks.
# Include details of company culture, customers and careers/jobs if you have the information.
# """


In [27]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [28]:
get_brochure_user_prompt("RAGHU ENGINEERING COLLEGE", "https://www.raghuenggcollege.com/")

'\nYou are looking at a company called: RAGHU ENGINEERING COLLEGE\nHere are the contents of its landing page and other relevant pages;\nuse this information to build a short brochure of the company in markdown without code blocks.\n\n\n## Landing Page:\n\nHome  - Raghu Engineering College - Autonomous | Vishakhapatnam\n\nSkip to content\nALUMNI\nAICTE\nNAAC\nIQAC\nIIC\nNBA\nNPTEL\nNIRF\nISO\nIIRS_ISRO\nICT_TOOLS\nMATLAB\nABC\nBCDE\nE_FEE_PAYMENTS\nDTBU\nHelp_Desk_91_8712497101\nRAGHU\nENGINEERING COLLEGE\n(AUTONOMOUS | VISAKHAPATNAM)\nAccredited by NAAC With ‘A+’ Grade\nAUTONOMOUS UGC 12(B) and 2(F)\nReaccredited by NBA to CIVIL, MECH, ECE & CSE\nPermanently Affiliated by JNTU-GV Vizianagaram\nApproved by All India Council for Technical Education (AICTE)\nCertified by ISO 14001:2015 & ISO 9001:2015\nCollege Code:\nRAGU\nAdmissions\n+91 9515127092\n+91 9885766942\n+91 08922 248001/2\nHome\nWho We Are\nAccreditations & Recognitions\nVision & Mission\nLeadership\nGoverning Body\nAcademic 

In [30]:
def create_brochure(company_name, url):
    response = gemini.chat.completions.create(
        model="gemini-3.5-flash-lite",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [31]:
create_brochure("Raghu Engineering College","https://www.raghuenggcollege.com/")

# Raghu Engineering College
### Autonomous | Visakhapatnam

Welcome to Raghu Engineering College (REC), the premier multi-disciplinary engineering institution in Andhra Pradesh. Established in 2001 and set across a sprawling 50-acre green campus, we are dedicated to delivering world-class technical education, fostering cutting-edge research, and shaping the next generation of industry leaders.

## Accreditation and Excellence
REC operates as an autonomous institution, proudly holding UGC 12(B) and 2(F) status and permanent affiliation with JNTU-GV Vizianagaram. Our commitment to academic excellence and institutional quality is reflected in our numerous recognitions:
* Accredited by NAAC with an ‘A+’ Grade
* Re-accredited by NBA for Civil, Mechanical, ECE, and CSE departments
* Approved by the All India Council for Technical Education (AICTE)
* Certified by ISO 14001:2015 & ISO 9001:2015 standards

## Academic Programs & Departments
We offer forward-thinking programs designed to meet the demands of modern industry. Our departments include:
* Computer Science & Engineering (with specialized tracks in AIML, Data Science, Cyber Security, and IoT)
* Electronics & Communications Engineering
* Mechanical Engineering
* Civil Engineering
* Electrical & Electronics Engineering
* Basic Science & Humanities

## Campus Culture & Student Life
At Raghu Engineering College, student life extends far beyond the classroom. Our vibrant campus culture encourages holistic development through:
* **Student Chapters & Clubs:** Fostering technical competence, leadership, and peer collaboration.
* **NCC & NSS:** Instilling social responsibility, discipline, and community service values.
* **Sports & Gymnasium:** Promoting physical fitness, teamwork, and athletic excellence.
* **World-Class Amenities:** Complete with 24/7 Wi-Fi availability, campus residency, secure transport, an on-site cafeteria, water treatment plant, ATM facility, and ample parking.

## For Prospective Students & Parents
Join a thriving academic community of over 8,910 students guided by 462 dedicated faculty members, including 103 doctorates. Our focused training model ensures that students build a strong academic and practical foundation from day one.

## Opportunities for Investors & Industry Partners
REC stands as a single destination for countless high-value opportunities. We maintain strong ties with the corporate world, serving as a reliable talent pipeline for top-tier global organizations looking for exceptionally trained engineering professionals.

## Impeccable Placements
Our dedicated campus recruitment process bridges the gap between academia and industry. 
* **6,887+** placement offers secured from 2022 to 2026
* **128+** global corporate recruiters
* Focused career training that consistently mines the most lucrative job opportunities and highest packages for our graduates.

## Careers & Joining Us
Are you passionate about shaping future engineers and advancing technological research? Raghu Engineering College offers a dynamic, supportive, and growth-oriented environment for educators and professionals. Explore our careers section to join our esteemed faculty and staff.

---
**Contact Us:**
* **College Code:** RAGU
* **Location:** Visakhapatnam, Andhra Pradesh
* **Admissions Helpline:** +91 9515127092 / +91 9885766942

In [ ]:
def create_brochure(company_name, url):
    response = gemini.chat.completions.create(
        model="gemini-3.5-flash-lite",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
        stream=True
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [33]:
create_brochure("Raghu Engineering College","https://www.raghuenggcollege.com/")

# RAGHU ENGINEERING COLLEGE
### Autonomous | Visakhapatnam

Welcome to Raghu Engineering College (REC), the premier multi-disciplinary engineering institution in Andhra Pradesh. Established in 2001 and set across a sprawling 50-acre green campus, we are committed to delivering world-class education, pioneering research, and unmatched career opportunities.

---

## Academic Excellence & Programs
REC is an autonomous institution recognized under UGC 12(B) and 2(F), permanently affiliated with JNTU-GV Vizianagaram, and approved by the AICTE. We hold a prestigious 'A+' grade accreditation from NAAC, and our core engineering programs (Civil, Mechanical, ECE, and CSE) are re-accredited by the NBA. 

We offer cutting-edge undergraduate and specialized programs across multiple departments:
* Computer Science & Engineering (including specializations in AI & ML, Data Science, Cyber Security, and IoT)
* Electronics & Communication Engineering
* Electrical & Electronics Engineering
* Mechanical Engineering
* Civil Engineering
* Basic Science & Humanities

---

## Exceptional Placements & Global Corporate Connect
For prospective students and investors looking at outcomes, REC is a single destination for countless opportunities. Backed by focused training and a strong academic foundation, our students are primed for lucrative careers:
* **6,887+** placement offers secured between 2022 and 2026.
* Partnerships with **128+ global corporates**.
* Leading the region in mining top-tier opportunities with industry-leading salary packages.

---

## Campus Culture & Student Life
At Raghu Engineering College, education extends far beyond the classroom. Our vibrant campus culture encourages holistic development through:
* Active student chapters and diverse student clubs.
* Comprehensive NCC and NSS units fostering social responsibility.
* Robust sports facilities and a fully-equipped gymnasium.

---

## World-Class Infrastructure & Amenities
Our 50-acre campus provides a secure, self-sustained ecosystem designed for optimal learning and living:
* Campus residency and reliable transportation facilities.
* 24/7 Wi-Fi connectivity across the grounds.
* Essential amenities including an on-site ATM, a dedicated water plant, a hygienic cafeteria, and secure 24/7 security with ample parking.

---

## Faculty & Careers
REC is proud to be powered by a dedicated team of **462 expert faculty members**, including **103 doctorates** committed to mentorship and academic rigor. 

For prospective recruits and faculty professionals seeking to shape the future of engineering, Raghu Engineering College offers a dynamic, research-driven work environment certified by ISO 9001:2015 and ISO 14001:2015. Join us in our mission to drive educational and technological excellence.

---

### Connect With Us
* **Location:** Visakhapatnam, Andhra Pradesh (College Code: RAGU)
* **Admissions Contact:** +91 9515127092 / +91 9885766942 / +91 08922 248001/2
* **Help Desk:** +91 8712497101